In [2]:
import math


class UAVConceptualSizer:

    def __init__(self):

        # =====================================================
        # MISSION REQUIREMENTS
        # =====================================================

        self.payload_kg = 10.0
        self.cruise_speed = 25.0          # m/s
        self.endurance_hr = 5.0
        self.altitude_m = 500.0

        # =====================================================
        # CONSTANTS
        # =====================================================

        self.g = 9.81
        self.rho_0 = 1.225
        self.T_0 = 288.15
        self.lapse_rate = 0.0065
        self.R = 287.05

        # =====================================================
        # DESIGN ASSUMPTIONS
        # =====================================================

        self.payload_fraction = 0.25
        self.wing_loading = 180.0         # N/m^2
        self.aspect_ratio = 12.0

        self.CL_max = 1.3
        self.CD_0 = 0.015
        self.oswald_efficiency = 0.85

        self.LD_target = 22.0

        # =====================================================
        # PROPULSION
        # =====================================================

        self.eta_prop = 0.80
        self.eta_motor = 0.85
        self.eta_esc = 0.95

        self.eta_total = (
            self.eta_prop *
            self.eta_motor *
            self.eta_esc
        )

        # =====================================================
        # BATTERY
        # =====================================================

        self.energy_density = 300.0       # Wh/kg
        self.system_voltage = 50.4        # 14S Li-Ion

    # =========================================================
    # AIR DENSITY
    # =========================================================

    def air_density(self):

        temp_ratio = (
            1 -
            (self.lapse_rate * self.altitude_m) /
            self.T_0
        )

        exponent = (
            (self.g / (self.R * self.lapse_rate))
            - 1
        )

        rho = self.rho_0 * math.pow(temp_ratio, exponent)

        return rho

    # =========================================================
    # MTOW
    # =========================================================

    def calculate_mtow(self):

        mtow_kg = (
            self.payload_kg /
            self.payload_fraction
        )

        weight_n = mtow_kg * self.g

        return mtow_kg, weight_n

    # =========================================================
    # WING SIZING
    # =========================================================

    def wing_geometry(self, weight_n):

        wing_area = (
            weight_n /
            self.wing_loading
        )

        wingspan = math.sqrt(
            self.aspect_ratio * wing_area
        )

        mac = wing_area / wingspan

        return wing_area, wingspan, mac

    # =========================================================
    # STALL SPEED
    # =========================================================

    def stall_speed(self, weight_n, rho, wing_area):

        v_stall = math.sqrt(
            (2 * weight_n) /
            (rho * wing_area * self.CL_max)
        )

        return v_stall

    # =========================================================
    # CRUISE AERODYNAMICS
    # =========================================================

    def aerodynamic_analysis(self, weight_n, rho, wing_area):

        CL = (
            (2 * weight_n) /
            (rho * self.cruise_speed**2 * wing_area)
        )

        CD = (
            self.CD_0 +
            (CL**2) /
            (
                math.pi *
                self.aspect_ratio *
                self.oswald_efficiency
            )
        )

        LD = CL / CD

        return CL, CD, LD

    # =========================================================
    # POWER ANALYSIS
    # =========================================================

    def power_analysis(self, weight_n, LD):

        thrust_required = weight_n / LD

        mechanical_power = (
            thrust_required *
            self.cruise_speed
        )

        electrical_power = (
            mechanical_power /
            self.eta_total
        )

        return (
            thrust_required,
            mechanical_power,
            electrical_power
        )

    # =========================================================
    # BATTERY ANALYSIS
    # =========================================================

    def battery_analysis(self, electrical_power):

        energy_required = (
            electrical_power *
            self.endurance_hr *
            1.20
        )

        capacity_ah = (
            energy_required /
            self.system_voltage
        )

        battery_mass = (
            energy_required /
            self.energy_density
        )

        return (
            energy_required,
            capacity_ah,
            battery_mass
        )

    # =========================================================
    # RANGE
    # =========================================================

    def range_analysis(self, endurance_hr):

        cruise_speed_kmh = (
            self.cruise_speed * 3.6
        )

        max_range = (
            cruise_speed_kmh *
            endurance_hr
        )

        return max_range

    # =========================================================
    # MAIN EXECUTION
    # =========================================================

    def run(self):

        rho = self.air_density()

        mtow_kg, weight_n = self.calculate_mtow()

        wing_area, wingspan, mac = (
            self.wing_geometry(weight_n)
        )

        v_stall = self.stall_speed(
            weight_n,
            rho,
            wing_area
        )

        CL, CD, LD = self.aerodynamic_analysis(
            weight_n,
            rho,
            wing_area
        )

        (
            thrust,
            mech_power,
            elec_power
        ) = self.power_analysis(weight_n, LD)

        (
            energy_req,
            capacity_ah,
            battery_mass
        ) = self.battery_analysis(elec_power)

        endurance_max = (
            energy_req / elec_power
        )

        max_range = self.range_analysis(
            endurance_max
        )

        # =====================================================
        # OUTPUT
        # =====================================================

        print("\n========== UAV CONCEPTUAL DESIGN ==========\n")

        print(f"Air Density              : {rho:.3f} kg/m^3")
        print(f"MTOW                     : {mtow_kg:.2f} kg")
        print(f"Aircraft Weight          : {weight_n:.2f} N")

        print("\n--- Wing Geometry ---")
        print(f"Wing Area                : {wing_area:.3f} m^2")
        print(f"Wingspan                 : {wingspan:.2f} m")
        print(f"Mean Aerodynamic Chord   : {mac:.3f} m")

        print("\n--- Aerodynamics ---")
        print(f"Stall Speed              : {v_stall:.2f} m/s")
        print(f"Cruise CL                : {CL:.3f}")
        print(f"Cruise CD                : {CD:.4f}")
        print(f"L/D Ratio                : {LD:.2f}")

        print("\n--- Propulsion ---")
        print(f"Required Thrust          : {thrust:.2f} N")
        print(f"Mechanical Power         : {mech_power:.2f} W")
        print(f"Electrical Power         : {elec_power:.2f} W")

        print("\n--- Battery System ---")
        print(f"Required Energy          : {energy_req:.2f} Wh")
        print(f"Battery Capacity         : {capacity_ah:.2f} Ah")
        print(f"Battery Mass             : {battery_mass:.2f} kg")

        print("\n--- Mission Performance ---")
        print(f"Maximum Endurance        : {endurance_max:.2f} hr")
        print(f"Maximum Range            : {max_range:.2f} km")

        print("\n===========================================\n")


if __name__ == "__main__":

    uav = UAVConceptualSizer()

    uav.run()


========== UAV CONCEPTUAL DESIGN ==========

Air Density              : 1.167 kg/m^3
MTOW                     : 40.00 kg
Aircraft Weight          : 392.40 N

--- Wing Geometry ---
Wing Area                : 2.180 m^2
Wingspan                 : 5.11 m
Mean Aerodynamic Chord   : 0.426 m

--- Aerodynamics ---
Stall Speed              : 15.40 m/s
Cruise CL                : 0.493
Cruise CD                : 0.0226
L/D Ratio                : 21.84

--- Propulsion ---
Required Thrust          : 17.97 N
Mechanical Power         : 449.26 W
Electrical Power         : 695.46 W

--- Battery System ---
Required Energy          : 4172.74 Wh
Battery Capacity         : 82.79 Ah
Battery Mass             : 13.91 kg

--- Mission Performance ---
Maximum Endurance        : 6.00 hr
Maximum Range            : 540.00 km


